# 04 Q1: ผู้โดยสารฟื้นตัวหลัง COVID แค่ไหน และช่องว่างอยู่ที่ไหน

Pain point ของ BART: รายได้ค่าโดยสารหายไป แต่ต้นทุนการเดินรถยังเท่าเดิม ยังไม่รู้ว่าควรคงหรือลดบริการตรงไหน

ข้อมูลที่ใช้
- marts จาก notebook 02 เฉพาะวัน normal
- วันทำงาน = จันทร์-ศุกร์ที่ไม่ใช่วันหยุดราชการ (นิยามเดียวกับ EDA)
- เทียบ ม.ค.-เม.ย. 2019 กับ ม.ค.-เม.ย. 2023
- recovery % = ค่าเฉลี่ยช่วง 2023 / ค่าเฉลี่ยช่วง 2019 × 100

ระดับรวมดูไปแล้วใน EDA (วันทำงาน 37%, วันหยุด 51%) notebook นี้เจาะว่าหายไปที่ไหน และฟื้นเร็วแค่ไหน

ข้อจำกัด: ข้อมูลไม่มีค่าโดยสาร จึงวัดได้แค่จำนวนทริป คำนวณรายได้ที่หายไปตรงๆ ไม่ได้

In [1]:
import sys
from pathlib import Path

if Path.cwd().name == "notebooks":
    ROOT = Path.cwd().parent
else:
    ROOT = Path.cwd()
sys.path.append(str(ROOT / "src"))

import pandas as pd
import plotly.graph_objects as go

from bart import viz
from bart.config import create_views, duck, fetch_row, fetch_value
from bart.perf import stage
from bart.periods import AFTER, BEFORE, period_case

viz.notebook_mode()

con = duck()
create_views(con)

## 1. Recovery index รายเดือน

ทริปเฉลี่ยต่อวันของแต่ละเดือน หารด้วยเดือนเดียวกันของปี 2019 (ตัดผลของฤดูกาลออก) แยกวันทำงานกับวันหยุด
เริ่มตั้งแต่ มิ.ย. 2020 (หลังช่วง break) ถึง เม.ย. 2023

In [2]:
recovery_query = """
    CREATE OR REPLACE TEMP TABLE recovery_index AS
    WITH monthly AS (
        SELECT
            year,
            month(date)    AS month_of_year,
            month,
            day_type,
            avg(day_trips) AS trips_per_day
        FROM calendar
        WHERE day_status = 'normal'
          AND month < DATE '2023-05-01'
        GROUP BY year, month(date), month, day_type
    ),
    base AS (
        SELECT month_of_year, day_type, trips_per_day AS trips_2019
        FROM monthly
        WHERE year = 2019
    )
    SELECT
        m.month,
        m.day_type,
        m.trips_per_day,
        b.trips_2019,
        100 * m.trips_per_day / b.trips_2019 AS recovery_index
    FROM monthly AS m
    JOIN base AS b ON b.month_of_year = m.month_of_year AND b.day_type = m.day_type
    WHERE m.month >= DATE '2020-06-01'
"""

with stage("04.recovery_index"):
    con.execute(recovery_query)

recovery = con.sql("SELECT * FROM recovery_index ORDER BY month, day_type").df()
recovery.pivot(index="month", columns="day_type", values="recovery_index").tail(12).round(1)

[perf] 04.recovery_index: 0.00s | 6.37 cores | peak RAM 0.209 GB


day_type,weekend_holiday,workday
month,,
2022-05-01,53.5,33.1
2022-06-01,52.3,34.2
2022-07-01,52.0,33.5
2022-08-01,54.4,35.2
2022-09-01,58.0,38.1
2022-10-01,60.6,37.9
2022-11-01,54.5,37.0
2022-12-01,50.7,36.7
2023-01-01,48.8,34.3


ความเร็วการฟื้นตัว: เทียบ index ช่วง ม.ค.-เม.ย. 2023 กับ ม.ค.-เม.ย. 2022 (ใช้ 4 เดือนเฉลี่ยกัน ไม่ใช้เดือนเดียว เพื่อลดความผันผวน)

In [3]:
pace_rows = []
for day_type in ["workday", "weekend_holiday"]:
    index_2022, index_2023 = fetch_row(con, f"""
        SELECT
            avg(recovery_index) FILTER (WHERE month BETWEEN DATE '2022-01-01' AND DATE '2022-04-01'),
            avg(recovery_index) FILTER (WHERE month BETWEEN DATE '2023-01-01' AND DATE '2023-04-01')
        FROM recovery_index
        WHERE day_type = '{day_type}'
    """)
    pace_rows.append({
        "day_type": day_type,
        "index_jan_apr_2022": round(index_2022, 1),
        "index_jan_apr_2023": round(index_2023, 1),
        "points_gained_in_12_months": round(index_2023 - index_2022, 1),
    })

pace = pd.DataFrame(pace_rows)
pace

,day_type,index_jan_apr_2022,index_jan_apr_2023,points_gained_in_12_months
0,workday,27.6,37.0,9.5
1,weekend_holiday,43.2,52.6,9.4


In [4]:
line_styles = {
    "workday": dict(color=viz.ACCENT, width=3),
    "weekend_holiday": dict(color=viz.CONTRAST, width=3),
}
line_names = {"workday": "Workday", "weekend_holiday": "Weekend / holiday"}

fig = go.Figure()
for day_type, style in line_styles.items():
    rows = recovery[recovery["day_type"] == day_type]
    fig.add_trace(go.Scatter(
        x=rows["month"],
        y=rows["recovery_index"],
        mode="lines",
        name=line_names[day_type],
        line=style,
        hovertemplate="%{x|%b %Y}: %{y:.0f}% of the same month in 2019<extra>%{fullData.name}</extra>",
    ))

fig.add_hline(y=100, line=dict(color=viz.MUTED, width=1, dash="dot"))
fig.update_yaxes(title_text="% of the same month in 2019", ticksuffix="%", range=[0, 105])

workday_pace = pace[pace["day_type"] == "workday"].iloc[0]
title = (
    f"Workday riders are at {workday_pace['index_jan_apr_2023']:.0f}% of 2019, "
    f"up only {workday_pace['points_gained_in_12_months']:.1f} points in a year"
)
subtitle = "Average trips per day each month as % of the same month in 2019, normal days, Jun 2020 to Apr 2023"
viz.titled(fig, title, subtitle)
viz.save(fig, "04_recovery_index")
fig

In [5]:
plateau = con.sql("""
    SELECT month, round(recovery_index, 1) AS workday_index
    FROM recovery_index
    WHERE day_type = 'workday' AND month >= DATE '2022-09-01'
    ORDER BY month
""").df()

plateau

,month,workday_index
0,2022-09-01,38.1
1,2022-10-01,37.9
2,2022-11-01,37.0
3,2022-12-01,36.7
4,2023-01-01,34.3
5,2023-02-01,37.6
6,2023-03-01,37.3
7,2023-04-01,38.9


index วันทำงานขึ้นเร็วช่วงครึ่งแรกของปี 2022 แต่ตั้งแต่ ก.ย. 2022 แทบไม่ขยับ อยู่ราว 37-39% มา 8 เดือน

### ตรวจซ้ำ: ถ้าเทียบกับระดับที่ควรเป็นถ้าไม่มี COVID

การเทียบกับปี 2019 สมมติว่าถ้าไม่มี COVID ผู้โดยสารจะเท่าปี 2019 ไปตลอด แต่ก่อน COVID ผู้โดยสารลดลงอยู่แล้วตั้งแต่ปี 2016
notebook 13 จึงสร้างเส้น counterfactual ด้วย SARIMA ที่เทรนเฉพาะข้อมูลก่อน COVID (ม.ค. 2011 ถึง ก.พ. 2020)

ผลช่วง ม.ค.-เม.ย. 2023
- เทียบกับปี 2019 (วิธีใน notebook นี้): 37.1%
- เทียบกับ counterfactual: 39.0% ช่วงความไม่แน่นอน 32-48%
- ลองโมเดลอื่นอีก 8 แบบ ได้ 37.1-39.6% ทุกแบบ

ตัวเลขใกล้กัน ข้อสรุปของ notebook นี้จึงไม่เปลี่ยน
notebook นี้ยังใช้การเทียบกับปี 2019 ต่อ เพราะต้องแยกรายวัน รายสถานี และราย corridor ซึ่ง counterfactual ทำได้แค่ยอดรวมทั้งระบบ

## 2. การฟื้นตัวแยกวันในสัปดาห์

ไม่นับวันหยุดราชการ ให้เหลือแค่ผลของวันในสัปดาห์

In [6]:
weekday_query = f"""
    WITH days_in_period AS (
        SELECT
            {period_case("date")} AS period,
            day_of_week,
            day_name,
            day_trips
        FROM calendar
        WHERE day_status = 'normal' AND NOT is_holiday
    ),
    by_day AS (
        SELECT period, day_of_week, day_name, avg(day_trips) AS trips_per_day, count(*) AS days
        FROM days_in_period
        WHERE period IS NOT NULL
        GROUP BY period, day_of_week, day_name
    )
    SELECT
        b.day_of_week,
        b.day_name,
        round(b.trips_per_day)                      AS trips_2019,
        round(a.trips_per_day)                      AS trips_2023,
        round(b.trips_per_day - a.trips_per_day)    AS lost_per_day,
        round(100 * a.trips_per_day / b.trips_per_day, 1) AS recovery_pct,
        b.days                                      AS days_2019,
        a.days                                      AS days_2023
    FROM by_day AS b
    JOIN by_day AS a ON a.day_of_week = b.day_of_week AND a.period = '{AFTER}'
    WHERE b.period = '{BEFORE}'
    ORDER BY b.day_of_week
"""

by_weekday = con.sql(weekday_query).df()
by_weekday

,day_of_week,day_name,trips_2019,trips_2023,lost_per_day,recovery_pct,days_2019,days_2023
0,1,Monday,401069.0,137966.0,263103.0,34.4,15,14
1,2,Tuesday,424303.0,160072.0,264231.0,37.7,17,17
2,3,Wednesday,415484.0,161437.0,254047.0,38.9,17,17
3,4,Thursday,422067.0,162976.0,259091.0,38.6,17,17
4,5,Friday,385319.0,134571.0,250748.0,34.9,17,17
5,6,Saturday,160742.0,83524.0,77219.0,52.0,16,17
6,7,Sunday,106961.0,59278.0,47682.0,55.4,17,17


In [7]:
colors = []
for day_of_week in by_weekday["day_of_week"]:
    if day_of_week <= 5:
        colors.append(viz.ACCENT)
    else:
        colors.append(viz.CONTRAST)

bars = go.Bar(
    x=by_weekday["day_name"],
    y=by_weekday["recovery_pct"],
    marker_color=colors,
    text=by_weekday["recovery_pct"].round(0).astype(int).astype(str) + "%",
    textposition="outside",
    customdata=by_weekday[["trips_2019", "trips_2023"]],
    hovertemplate="%{x}: %{customdata[1]:,.0f} vs %{customdata[0]:,.0f} trips per day<extra></extra>",
)

fig = go.Figure(bars)
fig.update_layout(hovermode="closest", showlegend=False)
fig.update_yaxes(title_text="% of Jan-Apr 2019", ticksuffix="%", range=[0, 70])

weakest_name = ""
weakest_pct = 101.0
strongest_name = ""
strongest_pct = -1.0
for _, row in by_weekday.iterrows():
    if row["day_of_week"] > 5:
        continue
    if row["recovery_pct"] < weakest_pct:
        weakest_name = row["day_name"]
        weakest_pct = row["recovery_pct"]
    if row["recovery_pct"] > strongest_pct:
        strongest_name = row["day_name"]
        strongest_pct = row["recovery_pct"]

title = (
    f"{weakest_name} recovered least ({weakest_pct:.0f}%), "
    f"{strongest_name} most ({strongest_pct:.0f}%) among workdays"
)
subtitle = "Trips per day in Jan-Apr 2023 as % of Jan-Apr 2019, normal non-holiday days. Blue = Mon-Fri, orange = weekend"
viz.titled(fig, title, subtitle)
viz.save(fig, "04_recovery_by_weekday")
fig

## 3. การฟื้นตัวราย corridor

ผู้โดยสารของสถานี = entries + exits ต่อวันทำงาน (เหมือน EDA) รวมเป็นราย corridor
MLPT และ BERY เปิดปี 2020 ไม่มีค่าปี 2019 จึงรวมในยอดปี 2023 ของ South Bay แต่ไม่มีฐานปี 2019 ให้เทียบ ตัด corridor นี้ออกจากการคำนวณ recovery %

In [8]:
station_query = f"""
    CREATE OR REPLACE TEMP TABLE station_recovery AS
    WITH days_in_period AS (
        SELECT
            d.station::VARCHAR        AS station,
            {period_case("d.date")} AS period,
            d.entries + d.exits       AS riders
        FROM station_day AS d
        JOIN calendar AS c ON c.date = d.date
        WHERE c.day_status = 'normal' AND c.is_workday
    ),
    by_station AS (
        SELECT station, period, avg(riders) AS riders
        FROM days_in_period
        WHERE period IS NOT NULL
        GROUP BY station, period
    )
    SELECT
        s.station,
        s.name,
        s.corridor,
        s.side,
        b.riders AS riders_2019,
        a.riders AS riders_2023
    FROM stations AS s
    LEFT JOIN by_station AS b ON b.station = s.station AND b.period = '{BEFORE}'
    LEFT JOIN by_station AS a ON a.station = s.station AND a.period = '{AFTER}'
"""

with stage("04.station_recovery"):
    con.execute(station_query)

corridors = con.sql("""
    WITH totals AS (
        SELECT
            corridor,
            sum(riders_2019) AS riders_2019,
            sum(riders_2023) AS riders_2023
        FROM station_recovery
        WHERE riders_2019 IS NOT NULL
        GROUP BY corridor
    )
    SELECT
        corridor,
        round(riders_2019)                                AS riders_2019,
        round(riders_2023)                                AS riders_2023,
        round(riders_2019 - riders_2023)                  AS lost_per_day,
        round(100 * riders_2023 / riders_2019, 1)         AS recovery_pct,
        round(100 * (riders_2019 - riders_2023) / sum(riders_2019 - riders_2023) OVER (), 1) AS pct_of_all_lost
    FROM totals
    ORDER BY recovery_pct
""").df()

corridors

[perf] 04.station_recovery: 0.01s | 1.96 cores | peak RAM 0.238 GB


,corridor,riders_2019,riders_2023,lost_per_day,recovery_pct,pct_of_all_lost
0,Dublin line,28898.0,9334.0,19564.0,32.3,3.8
1,SF Downtown,281651.0,91558.0,190092.0,32.5,36.5
2,Oakland Core,131968.0,49090.0,82879.0,37.2,15.9
3,Concord-Antioch line,83995.0,31334.0,52661.0,37.3,10.1
4,Fremont line,67971.0,26031.0,41941.0,38.3,8.1
5,SF Mission/South,86015.0,34116.0,51899.0,39.7,10.0
6,Peninsula,65156.0,26511.0,38645.0,40.7,7.4
7,Berkeley-Richmond line,74057.0,31641.0,42416.0,42.7,8.2


In [9]:
recovery_bars = go.Bar(
    y=corridors["corridor"],
    x=corridors["recovery_pct"],
    orientation="h",
    marker_color=viz.ACCENT,
    text=corridors["recovery_pct"].round(0).astype(int).astype(str) + "%",
    textposition="outside",
    hovertemplate="%{y}: %{x:.1f}% of 2019<extra></extra>",
)

fig = go.Figure(recovery_bars)
fig.update_layout(hovermode="closest", showlegend=False, margin=dict(l=200))
fig.update_xaxes(title_text="workday riders, % of Jan-Apr 2019", ticksuffix="%", range=[0, 55], showgrid=True, gridcolor=viz.GRID)
fig.update_yaxes(showgrid=False)

slowest = corridors.iloc[0]
fastest = corridors.iloc[-1]
title = f"No corridor is above half: {slowest['corridor']} is lowest ({slowest['recovery_pct']:.0f}%), {fastest['corridor']} highest ({fastest['recovery_pct']:.0f}%)"
subtitle = "Workday riders (entries + exits) in Jan-Apr 2023 as % of Jan-Apr 2019, by corridor"
viz.titled(fig, title, subtitle)
viz.save(fig, "04_recovery_by_corridor")
fig

In [10]:
by_lost = corridors.sort_values("lost_per_day", ascending=True)

lost_bars = go.Bar(
    y=by_lost["corridor"],
    x=by_lost["lost_per_day"],
    orientation="h",
    marker_color=viz.CONTRAST,
    text=by_lost["pct_of_all_lost"].round(0).astype(int).astype(str) + "% of all lost",
    textposition="outside",
    hovertemplate="%{y}: %{x:,.0f} fewer riders per workday<extra></extra>",
)

fig = go.Figure(lost_bars)
fig.update_layout(hovermode="closest", showlegend=False, margin=dict(l=200, r=40))
largest_loss = by_lost["lost_per_day"].max()
fig.update_xaxes(
    title_text="riders lost per workday (entries + exits)",
    tickformat="~s",
    showgrid=True,
    gridcolor=viz.GRID,
    range=[0, largest_loss * 1.3],
)
fig.update_yaxes(showgrid=False)

top_loss = by_lost.iloc[-1]
title = f"{top_loss['corridor']} alone accounts for {top_loss['pct_of_all_lost']:.0f}% of the riders lost"
subtitle = "Drop in average workday riders per corridor, Jan-Apr 2019 to Jan-Apr 2023"
viz.titled(fig, title, subtitle)
viz.save(fig, "04_lost_by_corridor")
fig

## 4. การฟื้นตัวรายสถานี

ดูว่าสถานีใหญ่ฟื้นช้ากว่าสถานีเล็กจริงไหม

In [11]:
stations_compared = con.sql("""
    SELECT
        station,
        name,
        corridor,
        riders_2019,
        riders_2023,
        100 * riders_2023 / riders_2019 AS recovery_pct
    FROM station_recovery
    WHERE riders_2019 IS NOT NULL
    ORDER BY recovery_pct
""").df()

size_correlation = fetch_value(con, """
    SELECT corr(rank_size, rank_recovery)
    FROM (
        SELECT
            rank() OVER (ORDER BY riders_2019)                 AS rank_size,
            rank() OVER (ORDER BY riders_2023 / riders_2019)   AS rank_recovery
        FROM station_recovery
        WHERE riders_2019 IS NOT NULL
    )
""")

print(f"rank correlation between station size and recovery: {size_correlation:.2f}")
print()
print("slowest 5:")
print(stations_compared.head(5)[["station", "name", "corridor", "recovery_pct"]].round(1).to_string(index=False))
print()
print("fastest 5:")
print(stations_compared.tail(5)[["station", "name", "corridor", "recovery_pct"]].round(1).to_string(index=False))

rank correlation between station size and recovery: -0.16

slowest 5:
station                              name             corridor  recovery_pct
   MONT                     Montgomery St          SF Downtown          26.8
   WARM        Warm Springs/South Fremont         Fremont line          26.9
   NCON            North Concord/Martinez Concord-Antioch line          27.5
   PHIL Pleasant Hill/Contra Costa Centre Concord-Antioch line          27.7
   MLBR                          Millbrae            Peninsula          29.2

fastest 5:
station            name               corridor  recovery_pct
   SANL     San Leandro           Fremont line          46.8
   OAKL Oakland Airport           Oakland Core          49.7
   RICH        Richmond Berkeley-Richmond line          50.3
   ANTC         Antioch   Concord-Antioch line          57.8
   SFIA     SFO Airport              Peninsula          59.6


In [12]:
corridor_order = []
for corridor in stations_compared["corridor"]:
    if corridor not in corridor_order:
        corridor_order.append(corridor)
corridor_order.sort()

labelled = set()
for station in stations_compared.head(4)["station"]:
    labelled.add(station)
for station in stations_compared.tail(4)["station"]:
    labelled.add(station)
for station in stations_compared.sort_values("riders_2019").tail(4)["station"]:
    labelled.add(station)

fig = go.Figure()
for position, corridor in enumerate(corridor_order):
    rows = stations_compared[stations_compared["corridor"] == corridor]
    labels = []
    for station in rows["station"]:
        if station in labelled:
            labels.append(station)
        else:
            labels.append("")
    fig.add_trace(go.Scatter(
        x=rows["riders_2019"],
        y=rows["recovery_pct"],
        mode="markers+text",
        name=corridor,
        text=labels,
        textposition="top center",
        textfont=dict(size=11, color=viz.INK2),
        marker=dict(color=viz.SERIES[position], size=11, line=dict(color="white", width=1)),
        customdata=rows[["station", "name"]],
        hovertemplate="%{customdata[0]} %{customdata[1]}<br>%{x:,.0f} riders in 2019, %{y:.0f}% recovered<extra>%{fullData.name}</extra>",
    ))

fig.update_layout(hovermode="closest", legend=dict(font=dict(size=11)), margin=dict(t=140))
fig.update_xaxes(title_text="workday riders in Jan-Apr 2019 (log scale)", type="log")
fig.update_yaxes(title_text="% of 2019 in Jan-Apr 2023", ticksuffix="%")

title = f"Station size barely explains recovery (rank r = {size_correlation:.2f}); airports and line ends did best"
subtitle = "Each dot is a station: workday riders in Jan-Apr 2019 (x, log scale) and % of that in Jan-Apr 2023 (y)"
viz.titled(fig, title, subtitle)
viz.save(fig, "04_station_recovery", height=600)
fig

## 5. ช่องว่างตามประเภทการเดินทาง

แบ่งทริปตามต้นทาง-ปลายทาง (ใช้ od_month วันทำงาน)
- to/from SF Downtown: ต้นทางหรือปลายทางเป็น EMBR, MONT, POWL, CIVC
- crossing the Bay: ข้ามอ่าว แต่ไม่เกี่ยวกับ downtown SF
- within east / within west: ไม่ข้ามอ่าว และไม่เกี่ยวกับ downtown SF

ค่าที่ได้คือทริปต่อวันทำงาน (นับทริปละครั้ง ไม่ใช่ entries + exits)

In [13]:
trip_type_query = f"""
    WITH workday_months AS (
        SELECT
            {period_case("o.month")} AS period,
            CASE
                WHEN so.corridor = 'SF Downtown' OR sd.corridor = 'SF Downtown' THEN 'to/from SF Downtown'
                WHEN so.side <> sd.side THEN 'crossing the Bay'
                WHEN so.side = 'east' THEN 'within east'
                ELSE 'within west'
            END AS trip_type,
            o.trips
        FROM od_month AS o
        JOIN stations AS so ON so.station = o.origin::VARCHAR
        JOIN stations AS sd ON sd.station = o.dest::VARCHAR
        WHERE o.day_type = 'workday'
    ),
    days_in_period AS (
        SELECT {period_case("month")} AS period, sum(days) AS days
        FROM month_days
        WHERE day_type = 'workday'
        GROUP BY period
    ),
    per_day AS (
        SELECT w.period, w.trip_type, sum(w.trips) / d.days AS trips_per_day
        FROM workday_months AS w
        JOIN days_in_period AS d ON d.period = w.period
        WHERE w.period IS NOT NULL
        GROUP BY w.period, w.trip_type, d.days
    )
    SELECT
        b.trip_type,
        round(b.trips_per_day)                              AS trips_2019,
        round(a.trips_per_day)                              AS trips_2023,
        round(b.trips_per_day - a.trips_per_day)            AS lost_per_day,
        round(100 * a.trips_per_day / b.trips_per_day, 1)   AS recovery_pct,
        round(100 * (b.trips_per_day - a.trips_per_day)
              / sum(b.trips_per_day - a.trips_per_day) OVER (), 1) AS pct_of_all_lost,
        round(100 * b.trips_per_day / sum(b.trips_per_day) OVER (), 1) AS pct_of_trips_2019
    FROM per_day AS b
    JOIN per_day AS a ON a.trip_type = b.trip_type AND a.period = '{AFTER}'
    WHERE b.period = '{BEFORE}'
    ORDER BY lost_per_day DESC
"""

trip_types = con.sql(trip_type_query).df()
trip_types

,trip_type,trips_2019,trips_2023,lost_per_day,recovery_pct,pct_of_all_lost,pct_of_trips_2019
0,to/from SF Downtown,275745.0,89821.0,185923.0,32.6,72.1,67.3
1,within east,79512.0,36502.0,43010.0,45.9,16.7,19.4
2,crossing the Bay,41325.0,17813.0,23512.0,43.1,9.1,10.1
3,within west,13273.0,7760.0,5513.0,58.5,2.1,3.2


In [14]:
type_bars = go.Bar(
    x=trip_types["trip_type"],
    y=trip_types["recovery_pct"],
    marker_color=viz.ACCENT,
    text=trip_types["recovery_pct"].round(0).astype(int).astype(str) + "%",
    textposition="outside",
    customdata=trip_types[["lost_per_day", "pct_of_all_lost"]],
    hovertemplate="%{x}: %{y:.1f}% recovered, %{customdata[0]:,.0f} trips lost per day (%{customdata[1]:.0f}% of all lost)<extra></extra>",
)

fig = go.Figure(type_bars)
fig.update_layout(hovermode="closest", showlegend=False)
fig.update_yaxes(title_text="workday trips, % of Jan-Apr 2019", ticksuffix="%", range=[0, 70])

downtown = trip_types[trip_types["trip_type"] == "to/from SF Downtown"].iloc[0]
title = f"Trips to and from downtown SF recovered least ({downtown['recovery_pct']:.0f}%) and make up {downtown['pct_of_all_lost']:.0f}% of the trips lost"
subtitle = "Workday trips per day in Jan-Apr 2023 as % of Jan-Apr 2019, by trip type"
viz.titled(fig, title, subtitle)
viz.save(fig, "04_recovery_by_trip_type")
fig

## สรุป: Insight → Pain point

pain point ของ BART ในข้อนี้มี 2 ส่วน คือ (ก) รายได้ค่าโดยสารที่หายไป และ (ข) ไม่รู้ว่าควรคงหรือลดบริการตรงไหน

| # | Insight (ตัวเลข) | Pain point | ตอบโจทย์แค่ไหน | ข้อเสนอ |
|---|---|---|---|---|
| 1 | วันทำงานอยู่ที่ 37% ของปี 2019 เพิ่มแค่ 9.5 จุดใน 12 เดือน และแทบไม่ขยับตั้งแต่ ก.ย. 2022 (38.1% เป็น 38.9%) | (ก) รายได้ | ตรงจุดเรื่องระดับและความเร็ว แต่บอกเป็นรายได้ไม่ได้ เพราะไม่มีข้อมูลค่าโดยสาร | วางแผนงบและบริการโดยถือว่า ~40% ของปี 2019 คือระดับปกติใหม่ระยะสั้น ไม่รอให้ผู้โดยสารกลับมาเอง |
| 2 | วันหยุดฟื้นได้ 52.6% สูงกว่าวันทำงาน (37.0%) และเพิ่มเร็วพอกัน (+9.4 จุด) | (ข) คงหรือลดบริการ | ตรงจุด | ไม่ควรลดบริการวันหยุดในสัดส่วนเดียวกับวันทำงาน เพราะเป็นกลุ่มที่ฟื้นดีกว่า |
| 3 | จันทร์ (34%) และศุกร์ (35%) ฟื้นน้อยกว่าอังคาร-พฤหัส (38-39%) วันจันทร์มีผู้โดยสารน้อยกว่าวันพุธ ~23,500 คน (-15%) | (ข) คงหรือลดบริการ | ตรงจุด | ใช้ขบวนสั้นลงหรือความถี่ต่ำลงวันจันทร์/ศุกร์ คงขนาดเต็มไว้วันอังคาร-พฤหัส |
| 4 | ทริปไป/กลับ downtown SF เคยเป็น 67% ของทริปวันทำงาน ฟื้นแค่ 33% และคิดเป็น 72% ของทริปที่หายไป | (ก) รายได้ | ตรงจุดเรื่องว่าช่องว่างอยู่ที่ไหน ส่วนรายได้ตอบได้บางส่วน | เน้นการตลาดและแพ็กเกจค่าโดยสารสำหรับคนทำงานแบบ hybrid ที่เข้า downtown 2-3 วันต่อสัปดาห์ เป็นข้อเสนอที่ข้อมูลชุดนี้ยังพิสูจน์ผลไม่ได้ |
| 5 | ทุก corridor ฟื้นได้แค่ 32-43% ไม่มี corridor ไหนเกินครึ่ง ขนาดสถานีอธิบายการฟื้นตัวได้น้อย (rank r = -0.16) สถานีสนามบินและปลายสายฟื้นดีที่สุด (SFIA 60%, ANTC 58%, OAKL 50%) | (ข) คงหรือลดบริการ | บางส่วน บอกได้ว่าไม่ควรตัดตาม corridor แต่ยังไม่ได้ดูรายชั่วโมง (ดู Q2) | ปรับบริการตามช่วงเวลาและวัน มากกว่าการตัดทั้ง corridor และรักษาบริการไปสนามบิน |
| 6 | WARM ฟื้นแค่ 27% ต่ำที่สุดนอก downtown และตอนเช้ากลายเป็นสถานีปลายทาง (EDA) | (ข) คงหรือลดบริการ | ยังไม่ตอบ เป็นสมมติฐานว่าผู้โดยสารย้ายไปใช้ MLPT/BERY ที่เปิดปี 2020 | ตรวจใน Q4 ก่อนตัดสินใจเรื่องบริการที่ WARM |

ข้อจำกัด
- ไม่มีข้อมูลค่าโดยสาร ทุกข้อวัดเป็นจำนวนทริป ถ้าจะแปลงเป็นรายได้ต้องใช้ตารางค่าโดยสารของ BART เพิ่ม
- ปี 2023 มีข้อมูล 4 เดือน ความเร็วการฟื้นตัวจึงเป็นภาพระยะสั้น
- MLPT และ BERY ไม่มีฐานปี 2019 จึงไม่ได้นับในการเทียบราย corridor

ต่อไป: 05_q2_peak_shape